In [3]:
import { ChromaClient } from 'chromadb'
import { OpenAIEmbeddingFunction } from '@chroma-core/openai'
import { RecursiveCharacterTextSplitter } from '@langchain/textsplitters'

// 这一格只做基础初始化：Chroma 客户端、集合名、数据路径、分块器。
const client = new ChromaClient()
const COLLECTION_NAME = 'python_tutorial_zh_cn_3_14'
const DATASET_ROOT = '../../data/python-tutorial-zh-cn-3.14'
const MANIFEST_PATH = `${DATASET_ROOT}/manifest.json`

// chunkSize / chunkOverlap 是最常见的 RAG baseline 配置，
// 先保证每块有足够上下文，再避免相邻块完全断开。
const splitter = new RecursiveCharacterTextSplitter({
  chunkSize: 800,
  chunkOverlap: 120,
})

function getCollection(collectionName: string) {
  // 这里把 embeddingFunction 绑定到 collection，后面 queryTexts 会自动做 embedding。
  const embedder = new OpenAIEmbeddingFunction({
    modelName: 'text-embedding-3-small',
  })

  return client.getOrCreateCollection({
    name: collectionName,
    embeddingFunction: embedder,
  })
}


In [4]:
type ManifestEntry = {
  order: number
  source_url: string
  local_html_path: string
  title: string
}

type ManifestFile = {
  entry_url: string
  requested_count: number
  actual_count: number
  generated_at: string
  pages: ManifestEntry[]
}

type SourceDoc = {
  id: string
  text: string
  metadata: {
    source_url: string
    local_html_path: string
    doc_title: string
    section_title: string
    order: number
  }
}

type ChunkDoc = {
  id: string
  text: string
  metadata: {
    source_url: string
    local_html_path: string
    doc_title: string
    section_title: string
    order: number
    chunk_index: number
  }
}

// 只解码这批文档里常见的实体，已经够我们做纯文本索引。
function decodeHtmlEntities(input: string): string {
  const namedEntities: Record<string, string> = {
    amp: '&',
    lt: '<',
    gt: '>',
    quot: '"',
    apos: "'",
    nbsp: ' ',
    mdash: '—',
    ndash: '–',
    hellip: '…',
    laquo: '«',
    raquo: '»',
  }

  return input.replace(/&(#x?[0-9a-f]+|[a-z]+);/gi, (_, entity: string) => {
    const lower = entity.toLowerCase()

    if (lower.startsWith('#x')) {
      return String.fromCodePoint(Number.parseInt(lower.slice(2), 16))
    }

    if (lower.startsWith('#')) {
      return String.fromCodePoint(Number.parseInt(lower.slice(1), 10))
    }

    return namedEntities[lower] ?? `&${entity};`
  })
}

// Python 文档正文基本都落在 .body[role=main] 这个容器里，
// 先把导航、页脚这些噪音裁掉，再做文本提取会更干净。
function extractMainHtml(html: string): string {
  const match = html.match(/<div class="body" role="main">([\s\S]*?)<div class="clearer"><\/div>/i)
  return match?.[1] ?? html
}

function extractFirstText(html: string, tagName: string): string {
  const pattern = new RegExp(`<${tagName}[^>]*>([\\s\\S]*?)<\\/${tagName}>`, 'i')
  const match = html.match(pattern)
  if (!match) {
    return ''
  }

  return stripHtml(match[1])
}

// 这里做的是一个轻量 HTML -> 纯文本清洗，目标是给 embedding 用，
// 不追求保留完整结构，只保留对检索最有用的文字内容。
function stripHtml(html: string): string {
  return decodeHtmlEntities(
    html
      .replace(/<script[\s\S]*?<\/script>/gi, ' ')
      .replace(/<style[\s\S]*?<\/style>/gi, ' ')
      .replace(/<noscript[\s\S]*?<\/noscript>/gi, ' ')
      .replace(/<svg[\s\S]*?<\/svg>/gi, ' ')
      .replace(/<img[^>]*>/gi, ' ')
      .replace(/<a[^>]*class="headerlink"[^>]*>[\s\S]*?<\/a>/gi, ' ')
      .replace(/<[^>]+>/g, ' ')
      .replace(/\s+/g, ' ')
      .trim(),
  )
}

async function loadPythonTutorialDocs(): Promise<SourceDoc[]> {
  // manifest 负责告诉我们：有哪些页面、原始 URL 是什么、本地文件在哪里。
  const manifest = JSON.parse(await Deno.readTextFile(MANIFEST_PATH)) as ManifestFile

  return await Promise.all(
    manifest.pages.map(async (page) => {
      const htmlPath = `${DATASET_ROOT}/${page.local_html_path}`
      const html = await Deno.readTextFile(htmlPath)
      const mainHtml = extractMainHtml(html)
      // h1 作为 section_title，后面检索结果展示时比只看文件名更友好。
      const sectionTitle = extractFirstText(mainHtml, 'h1') || page.title
      const text = stripHtml(mainHtml)

      return {
        id: page.local_html_path.replace(/[/.]/g, '-'),
        text,
        metadata: {
          source_url: page.source_url,
          local_html_path: page.local_html_path,
          doc_title: page.title,
          section_title: sectionTitle,
          order: page.order,
        },
      }
    }),
  )
}

async function buildChunkDocs(): Promise<ChunkDoc[]> {
  const sourceDocs = await loadPythonTutorialDocs()
  const chunkDocs: ChunkDoc[] = []

  for (const doc of sourceDocs) {
    // 一个 HTML 页面往往太长，直接整页 embedding 会让召回变钝，
    // 所以这里先拆成多个 chunk，再把 chunk 级元数据一并带上。
    const chunks = await splitter.splitText(doc.text)

    chunks.forEach((chunk, index) => {
      chunkDocs.push({
        id: `${doc.id}-chunk-${index}`,
        text: chunk,
        metadata: {
          ...doc.metadata,
          chunk_index: index,
        },
      })
    })
  }

  return chunkDocs
}


In [6]:
async function rebuildPythonTutorialIndex() {
  // 全量重建索引：读取本地 HTML -> 清洗 -> 分块 -> 写入 Chroma。
  const chunkDocs = await buildChunkDocs()

  try {
    await client.deleteCollection({ name: COLLECTION_NAME })
  } catch {
    // 第一次建库时这里报错是正常的，忽略就行
  }

  const collection = await getCollection(COLLECTION_NAME)

  // 这里只传 documents + metadatas，embedding 由 collection 的 embeddingFunction 自动生成。
  await collection.add({
    ids: chunkDocs.map((doc) => doc.id),
    documents: chunkDocs.map((doc) => doc.text),
    metadatas: chunkDocs.map((doc) => doc.metadata),
  })

  return {
    collection: COLLECTION_NAME,
    sourceDocs: new Set(chunkDocs.map((doc) => doc.metadata.source_url)).size,
    chunkCount: chunkDocs.length,
    preview: chunkDocs.slice(0, 3),
  }
}

// 运行这一行后，collection 里就是可以被 query 的 Python 教程索引。
const indexSummary = await rebuildPythonTutorialIndex()
indexSummary


{
  collection: "python_tutorial_zh_cn_3_14",
  sourceDocs: 17,
  chunkCount: 189,
  preview: [
    {
      id: "pages-tutorial-index-html-chunk-0",
      text: "Python 教程 小技巧 本教程被设计为针对新入门 Python 语言的 程序员 ，而 不是 新入门编程的 初学者 。 Python 是一门易于学习、功能强大的编程语言。它提供了高效的高级数据结构，还能简单有效地面向对象编程。Python 优雅的语法和动态类型以及解释型语言的本质，使它成为多数平台上写脚本和快速开发应用的理想语言。 Python 解释器及丰富的标准库针对所有主流系统平台以源代码或二进制形式在 Python 网站 https://www.python.org/ 上免费提供，并可自由分发。 该网站还包含许多免费的第三方 Python 模块、程序和工具以及附加文档的分享链接。 Python 解释器易于扩展，使用 C 或 C++（或其他 C 能调用的语言）即可为 Python 扩展新功能和数据类型。Python 也可用作定制软件中的扩展程序语言。 本教程非正式地介绍了 Python 语言和系统的基本概念和特性。 请注意它预期你对于编程的总体概念有基本的了解。 准备好一个 Python 解释器随时上手练习会很有帮助，但所有的示例都是完备自足的，因此本教程也可以离线阅读。 标准库与模块的内容详见 Python 标准库 。 Python 语言参考手册 是更正规的语言定义。如要编写 C 或 C++ 扩展请参考 扩展和嵌入 Python 解释器 和 Python/C API 参考手册 。此外，深入讲解 Python 的书籍也有很多。 本教程对每一个功能的介绍并不完整，甚至没有涉及全部常用功能，只是介绍了 Python 中最值得学习的功能，旨在让读者快速感受一下 Python 的特色。学完本教程的读者可以阅读和编写 Python 模块和程序，也可以继续学习 Python 标准库 。 强烈推荐阅读 术语对照表 。 1. 课前甜点 2. 使用 Python 的解释器 2.1.",
      metadata: {
        source_url:

In [7]:
type QueryHit = {
  text: string
  source_url: string
  section_title: string
  doc_title: string
  chunk_index: number
  distance: number | null
}

async function queryPythonTutorial(question: string, topK = 5): Promise<QueryHit[]> {
  const collection = await getCollection(COLLECTION_NAME)
  // 直接传 queryTexts，Chroma 会用同一个 embeddingFunction 生成查询向量。
  const result = await collection.query({
    queryTexts: [question],
    nResults: topK,
  })

  const documents = result.documents?.[0] ?? []
  const metadatas = result.metadatas?.[0] ?? []
  const distances = result.distances?.[0] ?? []

  // 把 Chroma 原始返回值整理成更适合 notebook 直接阅读的结构。
  return documents.map((text, index) => {
    const metadata = metadatas[index] as Record<string, string | number> | undefined

    return {
      text,
      source_url: String(metadata?.source_url ?? ''),
      section_title: String(metadata?.section_title ?? ''),
      doc_title: String(metadata?.doc_title ?? ''),
      chunk_index: Number(metadata?.chunk_index ?? -1),
      distance: distances[index] ?? null,
    }
  })
}

// 你后面只要换这里的问题，就能看到最相关的 5 个文档块。
await queryPythonTutorial('Python 里的 for 和 range 有什么区别？')


[
  {
    text: "可以不从 0 开始，且可以按给定的步长递增（即使是负数步长）： >>> list ( range ( 5 , 10 )) [5, 6, 7, 8, 9] >>> list ( range ( 0 , 10 , 3 )) [0, 3, 6, 9] >>> list ( range ( - 10 , - 100 , - 30 )) [-10, -40, -70] 要按索引迭代序列，可以组合使用 range() 和 len() ： >>> a = [ 'Mary' , 'had' , 'a' , 'little' , 'lamb' ] >>> for i in range ( len ( a )): ... print ( i , a [ i ]) ... 0 Mary 1 had 2 a 3 little 4 lamb 不过大多数情况下 enumerate() 函数很方便，详见 循环的技巧 。 如果直接打印一个 range 会发生意想不到的事情： >>> range ( 10 ) range(0, 10) range() 返回的对象在很多方面和列表的行为一样，但其实它和列表不一样。该对象只有在被迭代时才一个一个地返回所期望的列表项，并没有真正生成过一个含有全部项的列表，从而节省了空间。 这种对象称为可迭代对象 iterable ，适合作为需要获取一系列值的函数或程序构件的参数。 for 语句就是这样的程序构件；以可迭代对象作为参数的函数例如 sum() ： >>> sum ( range ( 4 )) # 0 + 1 + 2 + 3 6 后续我们会看到更多返回可迭代对象并以可迭代对象作为参数的函数。 在 数据结构 一章中，我们将讨论 list() 的更多细节。 4.4. break 和 continue 语句 break 语句将跳出最近的一层 for 或 while 循环:",
    source_url: "https://docs.python.org/zh-cn/3.14/tutorial/controlflow.html",
    section_title: "4. 更多控制流工具",
    doc_title: "4. 更多控制流工具",
    chunk_index: 2,
    distance: 0.3742